# UTS Data Mining
**Nama:** Sultan Abdul Fatah  **NIM:** 1003240033  **Kelas:** Informatika Eksekutif

> **Notebook ini sudah sesuai dengan kerangka laporan makalah**. Isi setiap bagian dengan kode dan penjelasan Anda sendiri. Sebelum dikumpulkan: *Runtime → Restart session and run all* harus berjalan tanpa error.

In [ ]:
import os
import kagglehub
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', 100)

# Dataset dari kagglehub
path = kagglehub.dataset_download("fedesoriano/stroke-prediction-dataset")
print("Path to dataset files:", path)

# Mencari file CSV di dalam folder dataset
csv_files = [f for f in os.listdir(path) if f.endswith('.csv')]

if not csv_files:
  raise FileNotFoundError("Error: No CSV file found in the downloaded dataset directory.")

csv_path = os.path.join(path, csv_files[0])

# Data mentah: JANGAN ditimpa
df_raw = pd.read_csv(csv_path)
df = df_raw.copy()
print("Dataset shape (rows, columns):", df.shape)

**Jawaban / catatan Anda:**

Shape: (5110, 12)

## Data Understanding

In [ ]:
# Tulis kode Anda di sini
display(df.head())

print("")
print("="*100)
print("Dataset Information:\n")
display(df.info())

## Data Profiling

**Pertanyaan pemandu:**
- Missing, duplikat, nilai tidak valid, inkonsistensi?

> Setiap temuan harus punya bukti dari kode.

In [ ]:
# Tulis kode Anda di sini

# Missing values
print("Missing Values\n")
missing_df = pd.DataFrame({
    'Missing Count': df.isnull().sum(),
    'Percentage (%)': (df.isnull().mean() * 100).round(2)
})
display(missing_df[missing_df['Missing Count'] > 0])

print("\n", "="*50, "\n")

# Duplicate
print("Duplicate\n")
print(f"Full row duplicates: {df.duplicated().sum()}")
print(f"Duplicate IDs: {df['id'].duplicated().sum()}")

print("\n", "="*50, "\n")

# Cek unique value dan kardinalitas kolom kategorikal
print("Categorical Value Counts\n")
cat_cols = ['gender', 'ever_married', 'work_type', 'Residence_type', 'smoking_status']
for col in cat_cols:
  print(f"col: {df[col].value_counts(dropna=False)}")

print("\n", "="*50, "\n")

# Cek konsistensi antar kolom dan nilai ekstrim
print("Konsistensi & Anomaly Checks\n")
print(f"Age < 18 AND ever_married == 'Yes': {len(df[(df['age'] < 18) & (df['ever_married'] == 'Yes')])}")
print(f"Work Type 'children' age range: {df[df['work_type'] == 'children']['age'].min(), "to", df[df['work_type'] == 'children']['age'].max()}")
print(f"Age < 10 & active/former smoker: {len(df[(df['age'] < 10) & (df['smoking_status'].isin(['smokes', 'formerly smoked']))])}")
print(f"Babies (age < 1 year) count: {len(df[df['age'] < 1])}")
print(f"Extreme BMI (> 60) count: {len(df[df['bmi'] > 60])}")

print("\n", "="*50, "\n")

# Distribusi numerik dan outlier (IQR)
print("Distribusi Numerik & IQR Outliers\n")
num_cols = ['age', 'avg_glucose_level', 'bmi']
display(df[num_cols].describe().T[['min', '25%', '50%', '75%', 'max', 'mean', 'std']])

for col in num_cols:
  Q1 = df[col].quantile(0.25)
  Q3 = df[col].quantile(0.75)
  IQR = Q3 - Q1
  lower, upper = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR
  outliers = df[(df[col] < lower) | (df[col] > upper)]
  print(f"{col}: skew={df[col].skew():.2f} | IQR outliers={len(outliers)} ({len(outliers)/len(df)*100:.2f}%) | bounds=[{lower:.2f}, {upper:.2f}]")

print("\n", "="*50, "\n")

# Distribusi target (stroke)
print("Target Distribusi\n")
target_df = pd.DataFrame({
    'Count': df['stroke'].value_counts(),
    'Percentage (%)': (df['stroke'].value_counts(normalize=True) * 100).round(2)
})
display(target_df)

print("\n", "="*50, "\n")

# Cek apakah BMI yang kosong berhubungan dengan stroke?
print("BMI MISSINGNESS VS STROKE RATE (%)\n")
display((pd.crosstab(df['bmi'].isnull(), df['stroke'], normalize='index') * 100).round(2))

##Data Cleaning

In [ ]:
# Tulis kode Anda di sini

df_clean = df_raw.copy()
print(f"Shape before cleaning\t\t: {df_clean.shape}")

# Delete kolom ID
df_clean = df_clean.drop(columns=['id'])

# Delete gender 'Other'
df_clean = df_clean[df_clean['gender'] != 'Other'].copy()

# Handle missing values pada kolom BMI
# Buat kolom penanda (flag) biar sinyal stroke rate tinggi (19.90%) tidak hilang
df_clean['bmi_is_missing'] = df_clean['bmi'].isnull().astype(int)

# Imputasi nilai BMI yang kosong menggunakan median berdasarkan gender
# (Female: 27.8, Male: 28.4)
df_clean['bmi'] = df_clean['bmi'].fillna(
    df_clean.groupby('gender')['bmi'].transform('median')
)

# Capping nilai ekstrem BMI > 60 menjadi 60.0 (13 baris)
# Nilai hingga 97.6 secara klinis sangat tidak wajar, berpotensi error input
df_clean['bmi'] = df_clean['bmi'].clip(upper=60.0)

# Validasi hasil cleaning
print(f"Shape after cleaning\t\t: {df_clean.shape}")
print(f"Remaining missing values\t\t: {df_clean.isnull().sum().sum()}")
print(f"Total stroke cases preserved\t: {df_clean['stroke'].sum()} of {df_raw['stroke'].sum()}")
print()
display(df_clean.head())

In [ ]:
# Simpan hasil (data mentah tetap utuh)
clean_file_name = 'stroke_data_clean.csv'
df_clean.to_csv(clean_file_name, index=False)
print(f"Clean dataset successfully saved to '{clean_file_name}'.")

## Bab 5. EDA

**Pertanyaan Pemandu:**
- Tulis pertanyaan analisis sebelum membuat grafik.
- Gunakan rate (persentase), bukan hanya jumlah.
- Mana Observation, Interpretation, Insight?
- Tuliskan hypothesis.

In [ ]:
# Tulis kode Anda di sini

# Setup
df_eda = df_clean.copy()
sns.set_theme(style='whitegrid')
overall_rate = df_eda['stroke'].mean() * 100

### Grafik Pertanyaan 1 (Kelompok Usia)

In [ ]:
# Stroke Rate berdasarkan Kelompok Usia

df_eda['age_group'] = pd.cut(
    df_eda['age'],
    bins = [0, 30, 45, 60, 75, 100],
    labels=['<30', '30-44', '45-59', '60-74', '75+'],
    right=False
)

age_stats = df_eda.groupby('age_group', observed=False)['stroke'].agg(['count', 'sum', 'mean']).reset_index()
age_stats['rate'] = age_stats['mean'] * 100

fig, ax = plt.subplots(figsize=(8, 4.5))
colors_1 = sns.color_palette('Reds', len(age_stats))
ax.bar(age_stats['age_group'].astype(str), age_stats['rate'], color=colors_1)
ax.axhline(overall_rate, color='black', linestyle='--', linewidth=1.2,
           label=f'Rata-rata Keseluruhan ({overall_rate:.2f}%)')

for i, row in age_stats.iterrows():
    ax.text(i, row['rate'] + 0.5, f"{row['rate']:.2f}%\n({int(row['sum'])}/{int(row['count'])})",
            ha='center', fontsize=9)

ax.set_title('Tingkat Kejadian Stroke (%) Berdasarkan Kelompok Usia', fontweight='bold')
ax.set_xlabel('Kelompok Usia (Tahun)')
ax.set_ylabel('Stroke Rate (%)')
ax.set_ylim(0, 24)
ax.legend()
plt.tight_layout()
plt.show()

### Grafik Pertanyaan 2 (Komorbiditas Hipertensi & Penyakit Jantung)

In [ ]:
# Stroke Rate berdasarkan Kombinasi Komorbiditas

def comorbidity_label(row):
    h, hd = row['hypertension'], row['heart_disease']
    if h == 0 and hd == 0:
        return 'Tidak Keduanya'
    elif h == 1 and hd == 0:
        return 'Hipertensi Saja'
    elif h == 0 and hd == 1:
        return 'Penyakit Jantung Saja'
    return 'Hipertensi + Jantung'

df_eda['comorbidity'] = df_eda.apply(comorbidity_label, axis=1)
order_com = ['Tidak Keduanya', 'Hipertensi Saja', 'Penyakit Jantung Saja', 'Hipertensi + Jantung']
com_stats = df_eda.groupby('comorbidity')['stroke'].agg(['count', 'sum', 'mean']).reindex(order_com).reset_index()
com_stats['rate'] = com_stats['mean'] * 100

fig, ax = plt.subplots(figsize=(8.5, 4.5))
colors_2 = sns.color_palette('OrRd', len(com_stats))
ax.bar(com_stats['comorbidity'], com_stats['rate'], color=colors_2)
ax.axhline(overall_rate, color='black', linestyle='--', linewidth=1.2,
           label=f'Rata-rata Keseluruhan ({overall_rate:.2f}%)')

for i, row in com_stats.iterrows():
    ax.text(i, row['rate'] + 0.5, f"{row['rate']:.2f}%\n({int(row['sum'])}/{int(row['count'])})",
            ha='center', fontsize=9, bbox=dict(facecolor='white', edgecolor='none', pad=1, alpha=0.8))

ax.set_title('Tingkat Kejadian Stroke (%) Berdasarkan Riwayat Hipertensi & Penyakit Jantung', fontweight='bold')
ax.set_xlabel('Kombinasi Riwayat Medis')
ax.set_ylabel('Stroke Rate (%)')
ax.set_ylim(0, 25)
ax.legend()
plt.tight_layout()
plt.show()

### Grafik Pertanyaan 3 (Kadar Gula Darah & BMI)

In [ ]:
# Kadar Gula Darah (Stroke Rate) dan Distribusi BMI

df_eda['glucose_cat'] = pd.cut(
    df_eda['avg_glucose_level'],
    bins=[0, 100, 126, 200, 300],
    labels=['Normal\n(<100)', 'Borderline\n(100-125)', 'Tinggi\n(126-199)', 'Sangat Tinggi\n(>=200)'],
    right=False
)
gluc_stats = df_eda.groupby('glucose_cat', observed=False)['stroke'].agg(['count', 'sum', 'mean']).reset_index()
gluc_stats['rate'] = gluc_stats['mean'] * 100

fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.5))

colors_3 = sns.color_palette('YlOrRd', len(gluc_stats))
axes[0].bar(gluc_stats['glucose_cat'].astype(str), gluc_stats['rate'], color=colors_3)
axes[0].axhline(overall_rate, color='black', linestyle='--', linewidth=1.2,
                label=f'Rata-rata ({overall_rate:.2f}%)')
for i, row in gluc_stats.iterrows():
    axes[0].text(i, row['rate'] + 0.4, f"{row['rate']:.2f}%\n({int(row['sum'])}/{int(row['count'])})",
                 ha='center', fontsize=8.5)
axes[0].set_title('Stroke Rate (%) per Kategori Kadar Gula Darah', fontweight='bold')
axes[0].set_xlabel('Kategori Rata-rata Glukosa (mg/dL)')
axes[0].set_ylabel('Stroke Rate (%)')
axes[0].set_ylim(0, 16)
axes[0].legend()

sns.boxplot(data=df_eda, x='stroke', y='bmi', color='#5c85b8', width=0.5, ax=axes[1])
axes[1].set_title('Distribusi BMI: Non-Stroke (0) vs Stroke (1)', fontweight='bold')
axes[1].set_xlabel('Status Stroke (0 = Tidak, 1 = Stroke)')
axes[1].set_ylabel('BMI')

plt.tight_layout()
plt.show()

### Grafik Pertanyaan 4 (Status Merokok & Cek Faktor Usia)

In [ ]:
# Status Merokok vs Stroke Rate & Distribusi Usia
smoke_order = ['formerly smoked', 'smokes', 'never smoked', 'Unknown']
smoke_stats = df_eda.groupby('smoking_status')['stroke'].agg(['count', 'sum', 'mean']).reindex(smoke_order).reset_index()
smoke_stats['rate'] = smoke_stats['mean'] * 100

fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.5))

colors_4 = sns.color_palette('Blues_r', len(smoke_stats))
axes[0].bar(smoke_stats['smoking_status'], smoke_stats['rate'], color=colors_4)
axes[0].axhline(overall_rate, color='black', linestyle='--', linewidth=1.2,
                label=f'Rata-rata ({overall_rate:.2f}%)')
for i, row in smoke_stats.iterrows():
    axes[0].text(i, row['rate'] + 0.25, f"{row['rate']:.2f}%\n({int(row['sum'])}/{int(row['count'])})",
                 ha='center', fontsize=8.5)
axes[0].set_title('Stroke Rate (%) Berdasarkan Status Merokok', fontweight='bold')
axes[0].set_xlabel('Status Merokok')
axes[0].set_ylabel('Stroke Rate (%)')
axes[0].set_ylim(0, 10)
axes[0].tick_params(axis='x', rotation=15)
axes[0].legend()

sns.boxplot(data=df_eda, x='smoking_status', y='age', order=smoke_order, color='#5c85b8', width=0.5, ax=axes[1])
axes[1].set_title('Distribusi Usia per Status Merokok (Cek Confounding)', fontweight='bold')
axes[1].set_xlabel('Status Merokok')
axes[1].set_ylabel('Usia (Tahun)')
axes[1].tick_params(axis='x', rotation=15)

plt.tight_layout()
plt.show()